In [ ]:
# “Adaptive Crypto Market Making System”

# Combine:

# market making
# inventory control
# microprice alpha
# queue modeling
# regime detection
# adverse selection avoidance

# That becomes:

# extremely interviewable
# extremely aligned with your narrative
# realistic
# technically deep
# trader-oriented

# IMPORTANT: Understand Binance Depth Updates

# The depth stream gives:

# incremental updates
# NOT:
# full order book snapshots

# Meaning:
# you must:

# get initial snapshot via REST
# apply websocket deltas sequentially

# This is CRITICAL.

# Otherwise your book becomes corrupted.

In [2]:
import json
import requests
import threading
import websocket
import numpy as np
from sortedcontainers import SortedDict
from collections import deque

class State:
    def __init__(self):
        self.bids = SortedDict()  # price -> size
        self.asks = SortedDict()
        self.my_bids = {}
        self.my_asks = {}
        
        # exchange truth
        self.market_book = {
            "bids": SortedDict(),
            "asks": SortedDict()
        }

        # execution simulator
        self.order_queues = {
            "bids": {},
            "asks": {}
        }

        self.bid_size = 0
        self.ask_size = 0
        self.bid_quote = 0.0
        self.ask_quote = 0.0

        # queue ahead of your order
        self.bid_queue_pos = 0
        self.ask_queue_pos = 0

        self.ewma_var = 0.0
        self.last_mid = 0.0

        # strategy/risk state
        self.inventory = 0
        self.cash = 0.0
        self.realized_pnl = 0.0
        self.unrealized_pnl = 0.0

        self.trade_imbalance = 0.0

        # state variables
        self.last_update_id = None
        self.buffer = []
        self.initialized = False

# def update_book(state, bids, asks):
#     for price, size in bids:
#         price = float(price)
#         size = float(size)

#         if size == 0:
#             state.bids.pop(price, None)
#         else:
#             state.bids[price] = size

#     for price, size in asks:
#         price = float(price)
#         size = float(size)

#         if size == 0:
#             state.asks.pop(price, None)
#         else:
#             state.asks[price] = size

def update_book(state, bids, asks):

    for price, size in bids:

        price = float(price)
        size = float(size)

        if size == 0:

            state.bids.pop(price, None)

            rebuild_price_level_queue(
                state,
                "bids",
                price,
                0
            )

        else:

            state.bids[price] = size

            rebuild_price_level_queue(
                state,
                "bids",
                price,
                size
            )

    for price, size in asks:

        price = float(price)
        size = float(size)

        if size == 0:

            state.asks.pop(price, None)

            rebuild_price_level_queue(
                state,
                "asks",
                price,
                0
            )

        else:

            state.asks[price] = size

            rebuild_price_level_queue(
                state,
                "asks",
                price,
                size
            )


import time
import uuid

class Order:
    def __init__(self,
        order_id,
        side,
        price,
        qty,
        remaining_qty,
        timestamp,
        owner):
        
        self.order_id: str
        self.side: str          # "BUY" or "SELL"
        self.price: float
        self.qty: float
        self.remaining_qty: float
        self.timestamp: float
        self.owner: str         # "self" or "market"

def create_order(side, price, qty, owner):
    return Order(
        order_id=str(uuid.uuid4()),
        side=side,
        price=price,
        qty=qty,
        remaining_qty=qty,
        timestamp=time.time(),
        owner=owner
    )

# Option A: naive volume fill
# fill_prob = trade_volume / (depth_at_price + epsilon)
# Option B: queue proxy model
# fill_prob = my_size / total_size_at_price
# Option C: toxicity-adjusted fill
# fill_prob *= (1 - order_flow_imbalance)

# def update_inventory(state, price, qty):
#     # check if trade hits YOUR quotes
#     if price <= state.bid_quote:
#         depth_at_price = state.bids.peekitem(-1)[1]
#     elif price >= state.ask_quote:
#         depth_at_price = state.asks.peekitem(0)[1]

#     fill_prob = qty / (depth_at_price + 1e-9) # epsilon -> a tiny number added to avoid division by zero

#     if random.random() < fill_prob:
#         if price <= state.bid_quote:
#             state.inventory += qty   # you got bought

#         elif price >= state.ask_quote:
#             state.inventory -= qty   # you got sold

#     print("inventory:", state.inventory)

# def update_inventory(state, price, qty):
#     # price = trade["price"]
#     # qty = trade["qty"]
#     print("Price: ", price, "BID_QUOTE:" , state.bid_quote, "ASK_QUOTE:", state.ask_quote)

#     # check if this trade is relevant to your quote
#     if price == state.bid_quote:
#         depth = state.bids.get(price, 0) # you want exact price-level liquidity

#         fill_prob = qty / (depth + 1e-9) # epsilon -> a tiny number added to avoid division by zero

#         if random.random() < fill_prob:
#             print('bid filled')
#             state.inventory += qty

#     elif price == state.ask_quote:
#         depth = state.asks.get(price, 0)

#         fill_prob = qty / (depth + 1e-9)

#         if random.random() < fill_prob:
#             print('ask filled')
#             state.inventory -= qty

def on_fill(state, price, qty, side):
    """
    side = 'BUY' or 'SELL' from your perspective
    """

    if side == "BUY":
        # you bought inventory
        state.inventory += qty
        state.cash -= price * qty

    elif side == "SELL":
        # you sold inventory
        state.inventory -= qty
        state.cash += price * qty


def compute_mid(state):
    best_bid = state.bids.peekitem(-1)[0]
    best_ask = state.asks.peekitem(0)[0]
    return (best_bid + best_ask) / 2


def compute_microprice(state):
    bid_price, bid_size = state.bids.peekitem(-1)
    ask_price, ask_size = state.asks.peekitem(0)
    return (ask_price * bid_size + bid_price * ask_size) / (bid_size + ask_size)


# 3. So your current model is actually:
# quote = microprice + inventory_skew

# This is already a simplified:

# Avellaneda–Stoikov market maker

# 3. Your inventory skew is too simplistic

# Right now:

# skew = -inventory * k

# This is directionally correct.

# But real MM systems scale inventory skew by:

# volatility
# spread
# inventory risk limit

# You want:

# reservation_price = fair - inventory * gamma * sigma²

# This is basically Avellaneda–Stoikov.

def compute_spread(state):
    sigma = get_vol(state)

    base = 0.03  # minimal spread

    vol_component = 3 * sigma  # scale factor

    return max(base, vol_component)

gamma = 0.1  # inventory risk aversion parameter

def compute_imbalance(state): # top of book imbalance
    bid_price, bid_size = state.bids.peekitem(-1)
    ask_price, ask_size = state.asks.peekitem(0)

    imbalance = (bid_size - ask_size) / (bid_size + ask_size + 1e-9)

    return imbalance


def compute_fair_price(state, micro):
    imbalance = compute_imbalance(state)
    flow = state.trade_imbalance

    # price impact from order flow
    alpha_imb = 0.2 # imbalance sensitivity
    alpha_flow = 0.05
    fair = micro + alpha_imb * imbalance + alpha_flow * flow

    return fair

def generate_quotes(state, k=0.5, spread=0.5):
    mid = compute_mid(state)
    micro = compute_microprice(state)

    #fair = micro  # or mix(mid, micro)
    fair = compute_fair_price(state)

    spread = compute_spread(state)

    #skew = -state.inventory * k

    skew = state.inventory * gamma * get_vol(state)  # Avellaneda–Stoikov, squared is dropped

    bid = fair - spread/2 + skew
    ask = fair + spread/2 + skew

    bid = round(bid, 2)
    ask = round(ask, 2)
    
    return bid, ask

TICK_SIZE = 0.01

def refresh_quotes(state):

    new_bid, new_ask = generate_quotes(state)

    new_bid = round(new_bid, 2)
    new_ask = round(new_ask, 2)

    old_bid = state.bid_quote
    old_ask = state.ask_quote

    # first placement
    if old_bid is None:

        place_quotes(state)
        return

    # only requote if sufficiently different
    bid_changed = abs(new_bid - old_bid) >= 2 * TICK_SIZE
    ask_changed = abs(new_ask - old_ask) >= 2 * TICK_SIZE

    if bid_changed or ask_changed:

        place_quotes(state)

def place_quotes(state):

    bid, ask = generate_quotes(state)

    size = 1.0

    # cancel previous quotes
    state.my_bids.clear()
    state.my_asks.clear()

    # place new bid
    state.my_bids[bid] = size

    # queue ahead of you
    state.bid_queue_pos = state.bids.get(bid, 0)

    # place new ask
    state.my_asks[ask] = size

    state.ask_queue_pos = state.asks.get(ask, 0)

    state.bid_quote = bid
    state.ask_quote = ask

def place_order(state, price, side, qty):
    book = state.order_queues[side]

    if price not in book:
        book[price] = deque()

    order = create_order(side, price, qty, owner="self")

    book[price].append(order)

# 9. Add cancel support (VERY interview relevant)
def cancel_order(state, order_id):

    for side in ["bids", "asks"]:

        for price, queue in state.order_queues[side].items():

            for order in list(queue):

                if order.order_id == order_id:
                    queue.remove(order)
                    return True

    return False

# 2. Faster / more “pro desk style”: exponential moving vol (EWMA)

# This is what most real systems use because it reacts quickly.

def update_vol(state):
    alpha = 0.94  # smoothing factor

    mid = compute_mid(state)

    if hasattr(state, "last_mid"):
        r = np.log(mid / state.last_mid)
        state.ewma_var = alpha * state.ewma_var + (1 - alpha) * r * r

    state.last_mid = mid

def get_vol(state):
    return np.sqrt(state.ewma_var)

def get_pnl(state):
    mid = compute_mid(state)

    unrealized = state.inventory * mid
    total_pnl = state.cash + unrealized

    return {
        "cash": state.cash,
        "inventory": state.inventory,
        "unrealized": unrealized,
        "total_pnl / mtm pnl": total_pnl
    }

alpha_flow = 0.2

def update_trade_flow(state, trade):
    side = trade["side"]

    flow = 1 if side == "BUY" else -1

    # This means:
    #     BUY pressure = upward pressure
    #     SELL pressure = downward pressure

    state.trade_imbalance = (
        alpha_flow * flow
        + (1 - alpha_flow) * state.trade_imbalance
    )

def compute_queue_ahead(state, side, price):
    book = state.order_queues[side]

    if price not in book:
        return 0

    queue = book[price]

    ahead = 0
    for order in queue:
        if order["is_me"]:
            break
        ahead += order["qty"]

    return ahead


def rebuild_price_level_queue(state, side, price, total_qty):

    book_side = state.order_queues[side]

    # preserve your own orders
    existing_self_orders = []

    if price in book_side:

        for order in book_side[price]:

            if order.owner == "self":
                existing_self_orders.append(order)

    # rebuild queue
    queue = deque()

    # synthetic market liquidity ahead of you
    if total_qty > 0:

        market_order = create_order(
            side="BUY" if side == "bids" else "SELL",
            price=price,
            qty=total_qty,
            owner="market"
        )

        queue.append(market_order)

    # your orders go behind market liquidity
    for order in existing_self_orders:
        queue.append(order)

    # overwrite queue
    book_side[price] = queue

In [3]:
state = State()

url = "https://api.binance.com/api/v3/depth?symbol=BTCUSDT&limit=1000"

snapshot = requests.get(url).json()
state.last_update_id = snapshot["lastUpdateId"]

print(snapshot.keys())

bids = dict(snapshot["bids"])
asks = dict(snapshot["asks"])

bids = {float(p): float(q) for p, q in snapshot["bids"]}
asks = {float(p): float(q) for p, q in snapshot["asks"]}

state.bids = SortedDict(bids)
state.asks = SortedDict(asks)

state.market_book["bids"] = SortedDict(bids)
state.market_book["asks"] = SortedDict(asks)

state.initialized = False

print("Bid:", bids)
print("Ask:", asks)

# bids:
# price → total resting buy liquidity at that price

# asks:
# price → total resting sell liquidity at that price

print(snapshot["bids"])
print(snapshot["lastUpdateId"])
print(bids)

dict_keys(['lastUpdateId', 'bids', 'asks'])
Bid: {75792.92: 4.2962, 75792.91: 0.00043, 75792.9: 0.00414, 75792.62: 7e-05, 75792.01: 0.00022, 75792.0: 0.09609, 75791.92: 0.00014, 75791.91: 0.03728, 75791.46: 0.00014, 75791.45: 0.26791, 75791.44: 0.30934, 75791.43: 7e-05, 75791.39: 0.00023, 75791.01: 7e-05, 75791.0: 0.004, 75790.97: 7e-05, 75790.02: 0.00669, 75790.0: 0.00862, 75789.99: 0.26705, 75789.5: 0.12756, 75789.31: 8e-05, 75789.3: 0.41385, 75789.11: 0.0066, 75789.1: 0.004, 75789.09: 0.04, 75788.91: 0.2764, 75788.9: 0.00714, 75788.8: 7e-05, 75788.79: 7e-05, 75788.78: 0.02862, 75788.58: 7e-05, 75788.01: 0.00283, 75788.0: 0.0084, 75787.85: 0.26776, 75787.84: 0.51369, 75787.59: 0.1145, 75787.58: 0.39789, 75787.26: 7e-05, 75787.2: 0.004, 75787.14: 0.01321, 75787.11: 0.0132, 75787.06: 0.0328, 75786.95: 0.02, 75786.34: 0.26777, 75786.33: 0.55484, 75786.09: 0.10205, 75786.0: 0.0084, 75785.95: 0.16814, 75785.34: 0.00707, 75785.3: 0.004, 75784.98: 7e-05, 75784.88: 0.00732, 75784.69: 7e-05, 

In [ ]:
# Now we have the initial snapshot loaded into our state.

# Depth layer -> pricing layer
def parse_book(message):
    bids = [(float(p), float(q)) for p, q in message.get("b", [])]
    asks = [(float(p), float(q)) for p, q in message.get("a", [])]

    return bids, asks

# def on_depth(state, message):
#     bids, asks = parse_book(message)
#     update_book(state, bids, asks)

#     refresh_quotes(state)
#     print("QUOTE_DEPTH:", state.bid_quote, state.ask_quote)

def on_depth(state, message):
    U = message["U"] # first update ID in the message
    u = message["u"] # last update ID in the message

    # gap detection
    if U > state.last_update_id + 1:
        print("GAP DETECTED — RESYNC REQUIRED")
        state.initialized = False
        return

    state.last_update_id = u

    bids, asks = parse_book(message)
    update_book(state, bids, asks)

    refresh_quotes(state)
    print("QUOTE_DEPTH:", state.bid_quote, state.ask_quote)

def depth_callback(ws, message):
    #print("RAW DEPTH MESSAGE RECEIVED")  # <- add this first
    msg = json.loads(message)
    on_depth(state, msg)

depth_socket = "wss://stream.binance.com:9443/ws/btcusdt@depth"

ws_depth = websocket.WebSocketApp(
    depth_socket,
    on_message=depth_callback
)

def parse_trade(message):
    print('trade_message', message)
    return {
        "side": "SELL" if message["m"] else "BUY", # message["m"] is a boolean flag called “isBuyerMaker”.
        "price": float(message["p"]),
        "qty": float(message["q"]),
        "timestamp": message["T"]
    }

# Trade messages tell you about actual trades that happened in the market.

# def on_trade(state, message):
#     trade = parse_trade(message)
#     side = trade["side"]
#     qty = trade["qty"]
#     price = trade["price"]

#     #update_inventory(state, price, qty)
#     on_fill(state, price, qty, side)

#     refresh_quotes(state)
#     print("QUOTE_TRADE:", state.bid_quote, state.ask_quote)

def try_initialize_book(state):
    for msg in state.buffer:
        U = msg["U"]
        u = msg["u"]

        # wait until we find first valid continuation
        if u <= state.last_update_id:
            continue

        if U <= state.last_update_id + 1 <= u:
            state.initialized = True
            print("BOOK SYNCED")
            on_depth(state, msg)
            return
        


# Trade Layer -> Execution Layer
def process_fills(state, trade):

    price = trade["price"]
    print('current price:', price, 'my_bid:', state.my_bids, 'my_ask:', state.my_asks)
    qty = trade["qty"]

    # BUY side (your bid gets hit)
    if price in state.order_queues["bids"]:
        queue = state.order_queues["bids"][price]

        while queue and qty > 0:
            order = queue[0]

            # current market order consumes entire resting order
            if order["qty"] <= qty:
                qty -= order["qty"]

                if order["is_me"]:
                    on_fill(state, price, order["qty"], "BUY")

                queue.popleft()

            # current market order partially fills resting order
            else:
                order["qty"] -= qty

                if order["is_me"]:
                    on_fill(state, price, qty, "BUY")

                qty = 0

    # buyer lifts your ask
    elif price in state.order_queues["asks"]:
        
        queue = state.order_queues["asks"][price]

        while queue and qty > 0:
            order = queue[0]

            # current market order consumes entire resting order
            if order["qty"] <= qty:
                qty -= order["qty"]

                if order["is_me"]:
                    on_fill(state, price, order["qty"], "SELL")

                queue.popleft()

            # current market order partially fills resting order
            else:
                order["qty"] -= qty

                if order["is_me"]:
                    on_fill(state, price, qty, "SELL")

                qty = 0


def process_trade(state, price, qty, side):

    book_side = "bids" if side == "SELL" else "asks"

    if price not in state.order_queues[book_side]:
        return

    queue = state.order_queues[book_side][price]

    while queue and qty > 0:

        order = queue[0]

        fill_size = min(order.remaining_qty, qty)

        order.remaining_qty -= fill_size
        qty -= fill_size

        # your own fill
        if order.owner == "self":

            on_fill(
                state,
                price,
                fill_size,
                order.side
            )

        # fully consumed
        if order.remaining_qty <= 0:
            queue.popleft()


def on_trade(state, message):

    trade = parse_trade(message)

    update_trade_flow(state, trade)

    process_fills(state, trade)

    pnl = get_pnl(state)

    print(
        "INV:",
        state.inventory,
        "PNL:",
        round(pnl["total_pnl / mtm pnl"], 2)
    )

def trade_callback(ws, message):
    msg = json.loads(message)
    on_trade(state, msg)

trade_socket = "wss://stream.binance.com:9443/ws/btcusdt@trade"

ws_trade = websocket.WebSocketApp(
    trade_socket,
    on_message=trade_callback
)

In [5]:
t1 = threading.Thread(target=ws_depth.run_forever)
t2 = threading.Thread(target=ws_trade.run_forever)

t1.start()
t2.start()

GAP DETECTED — RESYNC REQUIRED
GAP DETECTED — RESYNC REQUIRED
GAP DETECTED — RESYNC REQUIRED
current price: 75792.93 my_bid: {} my_ask: {}
INV: 0 PNL: 0.0
current price: 75792.93 my_bid: {} my_ask: {}
INV: 0 PNL: 0.0
current price: 75792.93 my_bid: {} my_ask: {}
INV: 0 PNL: 0.0
current price: 75792.93 my_bid: {} my_ask: {}
INV: 0 PNL: 0.0
current price: 75792.93 my_bid: {} my_ask: {}
INV: 0 PNL: 0.0
current price: 75792.93 my_bid: {} my_ask: {}
INV: 0 PNL: 0.0
current price: 75792.93 my_bid: {} my_ask: {}
INV: 0 PNL: 0.0
current price: 75792.93 my_bid: {} my_ask: {}
INV: 0 PNL: 0.0
current price: 75792.93 my_bid: {} my_ask: {}
INV: 0 PNL: 0.0
current price: 75792.93 my_bid: {} my_ask: {}
INV: 0 PNL: 0.0
current price: 75792.93 my_bid: {} my_ask: {}
INV: 0 PNL: 0.0
current price: 75792.93 my_bid: {} my_ask: {}
INV: 0 PNL: 0.0
current price: 75792.93 my_bid: {} my_ask: {}
INV: 0 PNL: 0.0
current price: 75792.93 my_bid: {} my_ask: {}
INV: 0 PNL: 0.0
current price: 75792.93 my_bid: {} my_a

In [70]:
def shutdown():
    print("Closing sockets...")
    ws_depth.close()
    ws_trade.close()

    t1.join()
    t2.join()

shutdown()

Closing sockets...


In [ ]:
def objective_function(state):
    pnl = get_pnl(state)
    inventory = state.inventory
    toxic_fills = state.adverse_rate  # proxy for toxic flow

    objective = pnl - 0.01 * inventory**2 - 0.5 * toxic_fills

# 4. Fourth: Add adversarial signal (THIS is what makes it interview-grade)

# Right now your system assumes:

# trades = noise + imbalance

# But in HFT interviews they want:

# “how do you detect when flow is toxic?”

# Add:

# Adverse selection proxy:
# price moves against your fill within N ms
# mid moves after fill

def update_adverse_selection(state, trade):
    post_mid = compute_mid(state)
    pre_mid = state.last_mid

    if trade["side"] == "BUY":
        adverse = post_mid < pre_mid
    else:
        adverse = post_mid > pre_mid

    state.adverse_rate = 0.99 * state.adverse_rate + 0.01 * adverse

# 5. Fifth: Add regime detection (simple but powerful)

# You already compute volatility — good.

def compute_regime(state):
    sigma = get_vol(state)

    low_threshold = 0.5
    high_threshold = 2.0

    if sigma < low_threshold:
        regime = "mean_reverting"

    elif sigma > high_threshold:
        regime = "trending"

    else:
        regime = "normal"

    state.regime = regime

0

In [ ]:
# # Notes

# quotes are not getting filled at all with quoting logic -> not competitive
# make spread adaptive

# 1. Your fill model is currently unrealistic

# This is the most important weakness.

# Right now:

# if price == state.bid_quote:

# This assumes:

# every trade at your price may fill you
# but ignores queue position

# In real HFT:

# queue position is EVERYTHING
# most passive orders never fill
# adverse selection dominates

In [ ]:
# 7. Honest ranking

# If you ship it as-is:

# 6.5–7/10 quant project

# If you add:

# PnL + fills
# evaluation metrics
# adverse selection

# Then:

# 8.5–9/10 quant internship-level project

# That’s where it becomes “this person can work on a prop desk codebase”.